In [ ]:
#Define Parameters

import numpy as np
import pandas as pd

import statsmodels
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint
from IPython.display import display

import matplotlib.pyplot as plt
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

import plotly.graph_objects as go
import yfinance as yf
from Quantapp.analytics import TimeSeriesAnalytics as Rolling
from Quantapp.Algorithm   import Algorithm
comp = Rolling()
algorithm = Algorithm()
#plt.rcParams["figure.figsize"] = (20, 7)

time_frame_week = 7
time_frame_short = 21
time_frame_mid   = 50
time_frame_long = 200


period     = '10y'

risk_free_rate = 0.02 / 252  # Annualized risk-free rate divided by trading days

def calculate_sortino_ratio(returns):
    excess_returns = returns - risk_free_rate
    downside_deviation = excess_returns[excess_returns < 0].std()
    sortino_ratio = excess_returns.mean() / downside_deviation if downside_deviation != 0 else np.nan
    return sortino_ratio

def calculate_risk_adjusted_returns(df, time_frame):
    daily_returns = df.pct_change()
    rolling_sortino_ratio = daily_returns.rolling(window=time_frame).apply(calculate_sortino_ratio)
    return rolling_sortino_ratio

def generate_series(arr):
    data = {}
    for ticker in arr:
        yf_ticker = ticker.replace('.', '-')
        data[ticker] = yf.Ticker(yf_ticker).history(period=period)['Close']
    df = pd.DataFrame(data)
    return df


def plot_returns(returns, time_frame):
    threshold = 0
    fig = px.line(returns , x=returns.index, y=returns.columns)
    fig.add_hline(y=threshold, line_dash="dash", line_color="red", annotation_text=f"y={threshold}")
    fig.add_hline(y=threshold, line_dash="dash", line_color="red", annotation_text=f"y={threshold}")
    fig.show()

    
def create_spreads(asset_series, benchmark_series, time_frame, mode='standard'):
    
    if mode == 'standard':
        asset_returns = asset_series.pct_change(time_frame)
        benchmark_returns= benchmark_series.pct_change(time_frame)
    elif mode == 'sortino':
        asset_returns = calculate_risk_adjusted_returns(asset_series, time_frame)
        benchmark_returns= calculate_risk_adjusted_returns(benchmark_series, time_frame)

    benchmark_minus_asset = asset_returns.apply(lambda x: benchmark_returns - x)
    benchmark_minus_asset.columns = ["Benchmark" + "_minus_" + col for col in benchmark_minus_asset.columns]
    return benchmark_minus_asset


def create_spread_plot(asset_spreads):
    spread_threshold = 0
    spread           = asset_spreads
    mean             = spread[spread>=0].mean()
    std_dev = spread[spread >= 0].std()
    #spread = asset_spreads[:200]


    fig = px.line(spread)
    fig.update_layout(title=asset_spreads.name)
    fig.add_hline(y=spread_threshold, line_dash="dash", line_color="red", annotation_text=f"y={spread_threshold}")
    fig.add_hline(y=mean , line_color="red", annotation_text="mean")
    fig.add_hline(y=mean + std_dev, line_dash="dash", line_color="blue", 
                  annotation_text="mean + 1 std dev", annotation_position="bottom right")
    fig.add_hline(y=mean - std_dev, line_dash="dash", line_color="blue", 
                  annotation_text="mean - 1 std dev", annotation_position="bottom right")
    fig.add_hline(y=mean + 2*std_dev, line_dash="dot", line_color="green", 
                  annotation_text="mean + 2 std dev", annotation_position="bottom right")
    fig.add_hline(y=mean - 2*std_dev, line_dash="dot", line_color="green", 
                  annotation_text="mean - 2 std dev", annotation_position="bottom right")
    fig.add_shape(type="rect",
                  xref="paper", yref="y",
                  x0=0, y0=mean, x1=1, y1=spread.max(),
                  fillcolor="green", opacity=0.2, line_width=0)
    #fig.update_layout(height=800)
    return fig

def create_side_by_side_subplots(fig1, fig2):
    fig = make_subplots(rows=1, cols=2, subplot_titles=(fig1.layout.title.text, fig2.layout.title.text))
    
    for trace in fig1.data:
        fig.add_trace(trace,row=1,col=1)

    for trace in fig2.data:
        fig.add_trace(trace,row=1,col=2)
    
    return fig

def plot_multiple_spreads(assets):
    for column in assets:
       asset_spreads = assets[column]
       create_spread_plot(asset_spreads).show()

def plot_risk_adjusted_returns(series,time_frame):
    series_adjusted_returns = calculate_risk_adjusted_returns(series,time_frame)
    negative_returns = series_adjusted_returns[series_adjusted_returns<0]
    mean = negative_returns.mean()
    standard_deviation = negative_returns.std()
    standard_deviation_level_three_fourths = mean - .5 * standard_deviation
    standard_deviation_level_single        = mean - standard_deviation

    fig = px.line(series_adjusted_returns)

    fig.add_hline(y=0, line_dash="dash", line_color="black", 
                annotation_text="Zero Line", annotation_position="bottom right")
    fig.add_hline(y=mean, line_dash="dot", line_color="blue", 
                annotation_text=f"Mean of negative returns: {mean:.2f}", annotation_position="top right")
    fig.add_hline(y=standard_deviation_level_three_fourths , line_dash="dashdot", line_color="red", 
                annotation_text=f".75 Std Dev: {standard_deviation_level_three_fourths :.2f}", annotation_position="top right")
    fig.add_hline(y=standard_deviation_level_single, line_dash="dashdot", line_color="red", 
                annotation_text=f"1 Std Dev: {standard_deviation_level_single:.2f}", annotation_position="top right")

    fig.add_shape(
        type="rect",
        x0=series_adjusted_returns.index.min(),
        x1=series_adjusted_returns.index.max(),
        y0=standard_deviation_level_three_fourths,
        y1=standard_deviation_level_single,
        fillcolor="green",
        opacity=0.2,
        line_width=0,
    )

    return fig
def filter_assets_by_positive_spread_std(asset_spreads):
    spreads = asset_spreads
    positive_spreads = spreads[spreads >= 0] 
    
    mean = positive_spreads.mean()
    std_dev = positive_spreads.std()

    latest_spread = spreads.iloc[-1]
    threshold = mean + std_dev

    return latest_spread>=threshold


def filter_assets_below_negative_std(asset_spreads):
    if not isinstance(asset_spreads, pd.Series):
        raise TypeError("asset_spreads must be a pandas Series")

    negative_spreads = asset_spreads[asset_spreads < 0]
    if negative_spreads.empty:
        return pd.Series(dtype=bool)  
    
    mean_negative = negative_spreads.mean()
    std_dev_negative = negative_spreads.std()

    threshold_negative = mean_negative - 0.75 * std_dev_negative
    return asset_spreads < threshold_negative

def get_sector_info(ticker):
    try:
        stock = yf.Ticker(ticker)
        sector = stock.info.get('sector', 'N/A')
        sub_industry = stock.info.get('industry', 'N/A')
        return {'Ticker': ticker, 'Sector': sector, 'Sub-Industry': sub_industry}
    except Exception as e:
        print(f"Error fetching data for {ticker}: {e}")
        return {'Ticker': ticker, 'Sector': 'N/A', 'Sub-Industry': 'N/A'}

    


SPY = 'SPY'
IWM = 'IWM'
QQQ = 'QQQ'
DIA = 'DIA'
TLT = 'TLT'
AGG = 'AGG'
GLD = 'GLD'
XLK = 'XLK'
XLF = 'XLF'
benchmark = SPY


INDICES          = ['SPY','QQQ','DIA','IWM']
SECTORS          = ['SPY','XLF','XLK','XLV','XLC','XLI','XLU','XLB','VNQ','XLP','XLY','XBI','XLE']
INDUSTRIES       = ['SPY', 'SMH', 'KRE','KIE', 'KBE']
TOP_SPY_HOLDINGS = ['SPY','MSFT', 'AAPL', 'NVDA', 'AMZN', 'META', 'GOOG', 'BRK-B', 'AVGO','JPM','XOM','TSLA','UNH','V']
TOP_QQQ_HOLDINGS = ['QQQ','MSFT', 'AAPL', 'NVDA', 'AMZN', 'META', 'GOOG', 'BRK-B', 'AVGO', 'COST', 'TSLA', 'NFLX','PEP','AMD','ADBE','LIN',
                    'QCOM','CSCO','TMUS','INTU','AMAT','TXN','AMGN','CMCSA', 'MU']
TOP_DIA_HOLDINGS = ['DIA','UNH','GS','MSFT','CAT','HD', 'AMGN', 'V', 'CRM', 'MCD', 'AXP','TRV','HON','JPM','AMZN',
                    'AAPL','BA','IBM','PG','CVX','JNJ','MRK','DIS','MMM','NKE','KO','WMT','DOW','CSCO','VZ','INTC']
TOP_XLK_HOLDINGS = ['XLK','MSFT','AAPL','AVGO','NVDA','CRM','AMD','ADBE','QCOM','CSCO','ACN','ORCL','INTU','AMAT','TXN',
                    'IBM','NOW', 'MU', 'INTC', 'LRCX','ADI','KLAC','PANW']
TOP_XLF_HOLDINGS = ['XLF','BRK-B', 'JPM','V','MA','BAC','WFC','GS','SPGI','AXP','PGR','MS','C','SCHW','CB','MMC','FI','BX','ICE','CME','PYPL','USB',
                    'PNC','MCO','AON','AIG']
BONDS            = ['AGG','IEF','TLT', 'HYG','LQD','TIPS', 'BKLN']
PRECIOUS_METALS  = ['GLD','SLV','GDX','XME']
CRYPTO           = ['BITO','BLOK']
ENERGY           = ['USO','UNG','OIH','XOP','TAN','ICLN','URA','URNM','GUSH','KOLD']
CAPITALIZATIONS  = ['SPY', 'IJH' , 'IJR']
INNOVATION       = ['ARKG','ARKF','ARKK']
LONG_LEVERAGE    = ['TQQQ','SOXL','SPXL','TNA','BOIL','NUGT','ERX','DPST']
SHORT_LEVERAGE   = ['SQQQ','SPXS','UDOW','SSO','TECL','FAS','NVDA','TQQQ', 'VXX','UVXY','VIXY','UVIX','SVXY','SOXS','TZA','USD','TSLL','LABU','DPST','NUGT','CONL']
FOREIGN_MARKETS  = ['EWZ','EWJ','EWA','EWG','EWW','EEM','EFA','FEZ','INDA','EWU','EWG']


sp500_url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
dow_url = 'https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average'
nasdaq_url = 'https://en.wikipedia.org/wiki/NASDAQ-100'


def get_general_info(table):
    tickers = table['Symbol'].tolist()
    tickers = ['BRK-B'if symbol == 'BRK.B' else symbol for symbol in tickers]
    tickers = ['BF-B'if symbol == 'BF.B' else symbol for symbol in tickers]

    sectors = []
    sub_industry = []
    market_cap = []

    for ticker in tickers:

        info = get_sector_info(ticker)
        sectors.append(info['Sector'])
        sub_industry.append(info['Sub-Industry'])
        market_cap.append(yf.Ticker(ticker).info.get('marketCap'))
    
    table['Sector'] = sectors
    table['Sub-Industry'] = sub_industry
    table['Market Cap'] = market_cap

    return table

tables = pd.read_html(sp500_url)
sp500_table = tables[0]
sp500_table['Symbol'].tolist()


tables = pd.read_html(sp500_url)
xlk_table = tables[0]
xlk_table = xlk_table[xlk_table['GICS Sector'] == 'Information Technology']

tables = pd.read_html(sp500_url)
xlf_table = tables[0]
xlf_table = xlk_table[xlk_table['GICS Sector'] == 'Financials']

tables = pd.read_html(sp500_url)
xlv_table = tables[0]
xlv_table = xlv_table[xlv_table['GICS Sector'] == 'Health Care']

tables = pd.read_html(sp500_url)
xli_table = tables[0]
xli_table = xli_table[xli_table['GICS Sector'] == 'Industrials']

tables = pd.read_html(sp500_url)
xly_table = tables[0]
xly_table = xly_table[xly_table['GICS Sector'] == 'Consumer Discretionary']

tables = pd.read_html(sp500_url)
xle_table = tables[0]
xle_table = xle_table[xle_table['GICS Sector'] == 'Energy']

tables = pd.read_html(sp500_url)
xlb_table = tables[0]
xlb_table = xlb_table[xlb_table['GICS Sector'] == 'Materials']


tables = pd.read_html(sp500_url)
xlc_table = tables[0]
xlc_table = xlc_table[xlc_table['GICS Sector'] == 'Communication Services']


tables = pd.read_html(sp500_url)
xlre_table = tables[0]
xlre_table = xlre_table[xlre_table['GICS Sector'] == 'Real Estate']

tables = pd.read_html(sp500_url)
xlc_table = tables[0]
xlc_table = xlc_table[xlc_table['GICS Sector'] == 'Communication Services']

tables = pd.read_html(sp500_url)
xlp_table = tables[0]
xlp_table = xlp_table[xlp_table['GICS Sector'] == 'Consumer Staples']

tables = pd.read_html(sp500_url)
xlu_table = tables[0]
xlu_table = xlu_table[xlu_table['GICS Sector'] == 'Utilities']


tables = pd.read_html(nasdaq_url)
qqq_table = tables[4]

tables = pd.read_html(dow_url)
dia_table = tables[1]
dia_table


INDICES          = ['SPY','QQQ','DIA','IWM']
SECTORS          = ['SPY','XLF','XLK','XLV','XLC','XLI','XLU','XLB','VNQ','XLP','XLY','XBI','XLE']
INDUSTRIES       = ['SPY', 'SMH', 'KRE','KIE', 'KBE']
SPY_HOLDINGS = sp500_table['Symbol'].tolist()
TOP_SPY_HOLDINGS = ['SPY','MSFT', 'AAPL', 'NVDA', 'AMZN', 'META', 'GOOG', 'BRK-B', 'AVGO','JPM','XOM','TSLA','UNH','V']
TOP_QQQ_HOLDINGS = ['QQQ','MSFT', 'AAPL', 'NVDA', 'AMZN', 'META', 'GOOG', 'BRK-B', 'AVGO', 'COST', 'TSLA', 'NFLX','PEP','AMD','ADBE','LIN',
                    'QCOM','CSCO','TMUS','INTU','AMAT','TXN','AMGN','CMCSA', 'MU']
QQQ_HOLDINGS     = qqq_table['Ticker'].tolist()
TOP_DIA_HOLDINGS = ['DIA','UNH','GS','MSFT','CAT','HD', 'AMGN', 'V', 'CRM', 'MCD', 'AXP','TRV','HON','JPM','AMZN',
                    'AAPL','BA','IBM','PG','CVX','JNJ','MRK','DIS','MMM','NKE','KO','WMT','DOW','CSCO','VZ','INTC']
DIA_HOLDINGS     = dia_table['Symbol'].tolist()
TOP_XLK_HOLDINGS = ['XLK','MSFT','AAPL','AVGO','NVDA','CRM','AMD','ADBE','QCOM','CSCO','ACN','ORCL','INTU','AMAT','TXN',
                    'IBM','NOW', 'MU', 'INTC', 'LRCX','ADI','KLAC','PANW']
XLK_HOLDINGS = xlk_table['Symbol'].tolist()
TOP_XLF_HOLDINGS = ['XLF','BRK-B', 'JPM','V','MA','BAC','WFC','GS','SPGI','AXP','PGR','MS','C','SCHW','CB','MMC','FI','BX','ICE','CME','PYPL','USB',
                    'PNC','MCO','AON','AIG']
XLF_HOLDINGS     = xlf_table['Symbol'].tolist()
XLI_HOLDINGS     = xli_table['Symbol'].tolist()
XLV_HOLDINGS     = xlv_table['Symbol'].tolist()
XLU_HOLDINGS     = xlu_table['Symbol'].tolist()
XLF_HOLDINGS     = xlf_table['Symbol'].tolist()
XLB_HOLDINGS     = xlb_table['Symbol'].tolist()
XLY_HOLDINGS     = xly_table['Symbol'].tolist()
XLRE_HOLDINGS    = xlre_table['Symbol'].tolist()
XLC_HOLDINGS     = xlc_table['Symbol'].tolist()
XLE_HOLDINGS     = xle_table['Symbol'].tolist()
XLP_HOLDINGS     = xlp_table['Symbol'].tolist()
BONDS            = ['AGG','IEF','TLT', 'HYG','LQD','TIPS', 'BKLN']
PRECIOUS_METALS  = ['GLD','SLV','GDX','XME']
CRYPTO           = ['GBTC','BLOK']
ENERGY           = ['USO','UNG','OIH','XOP','TAN','ICLN','URA','URNM','GUSH','KOLD']
CAPITALIZATIONS  = ['SPY', 'IJH' , 'IJR']
INNOVATION       = ['ARKG','ARKF','ARKK']
LONG_LEVERAGE    = ['TQQQ','SOXL','SPXL','TNA','BOIL','NUGT','ERX','DPST']
SHORT_LEVERAGE   = ['SQQQ','SPXS','UDOW','SSO','TECL','FAS','NVDA','TQQQ', 'VXX','UVXY','VIXY','UVIX','SVXY','SOXS','TZA','USD','TSLL','LABU','DPST','NUGT','CONL']
FOREIGN_MARKETS  = ['EWZ','EWJ','EWA','EWG','EWW','EEM','EFA','FEZ','INDA','EWU','EWG']



In [ ]:
#Load Data: Company Info

tables = pd.read_html(sp500_url)
sp500_table = tables[0]
sp500_table = sp500_table[['Symbol', 'GICS Sector', 'GICS Sub-Industry']]
sp500_table = sp500_table.rename(columns={'GICS Sector' : 'Sector', "GICS Sub-Industry": 'Sub-Industry'})

tables = pd.read_html(nasdaq_url)
nasdaq_table = tables[4]
nasdaq_table = nasdaq_table[['Ticker', 'GICS Sector', 'GICS Sub-Industry']]
nasdaq_table = nasdaq_table.rename(columns={'Ticker' : 'Symbol','GICS Sector' : 'Sector', "GICS Sub-Industry": 'Sub-Industry'})

tables = pd.read_html(dow_url)
dow_table = tables[1]
dow_table = dow_table[['Symbol', 'Industry']]
dow_tickers = dow_table['Symbol'].tolist()

dow_info    = get_general_info(dow_table)
nasdaq_info = get_general_info(nasdaq_table)
sp500_info  = get_general_info(sp500_table)
xlk_info    = sp500_info[ sp500_info['Sector'] == 'Technology']
xlf_info    = sp500_info[sp500_info['Sector'] == 'Financial Services']
xli_info    = sp500_info[ sp500_info['Sector'] == 'Industrials']
xlv_info    = sp500_info[sp500_info['Sector'] == 'Healthcare']
xlu_info    = sp500_info[ sp500_info['Sector'] == 'Utilities']
xlb_info    = sp500_info[sp500_info['Sector'] == 'Basic Materials']
xly_info    = sp500_info[ sp500_info['Sector'] == 'Consumer Cyclical']
xlc_info    = sp500_info[sp500_info['Sector'] == 'Communication Services']
xle_info    = sp500_info[ sp500_info['Sector'] == 'Energy']
xlre_info    = sp500_info[ sp500_info['Sector'] == 'Real Estate']
xlp_info    = sp500_info[ sp500_info['Sector'] == 'Consumer Defensive']

In [ ]:
#Load Data: Prices & Calculations

indices_series             = generate_series(INDICES)
sectors_series             = generate_series(SECTORS)
industries_series          = generate_series(INDUSTRIES)
top_spy_holdings_series    = generate_series(TOP_SPY_HOLDINGS)
spy_holdings_series        = generate_series(top_spy_holdings_series)
top_qqq_holdings_series    = generate_series(TOP_QQQ_HOLDINGS)
qqq_holdings_series        = generate_series(QQQ_HOLDINGS)
top_dia_holdings_series    = generate_series(TOP_DIA_HOLDINGS)
dia_holdings_series        = generate_series(DIA_HOLDINGS)
top_xlk_holdings_series    = generate_series(TOP_XLK_HOLDINGS)
xlk_holdings_series        = generate_series(XLK_HOLDINGS)
top_xlf_holdings_series    = generate_series(TOP_XLF_HOLDINGS) 
xlf_holdings_series        = generate_series(XLF_HOLDINGS)
benchmark_series           = indices_series[benchmark]

all_series = pd.concat([
    indices_series,
    sectors_series,
    industries_series,
    top_spy_holdings_series,
    top_qqq_holdings_series,
    top_dia_holdings_series,
    top_xlk_holdings_series,
    top_xlf_holdings_series,
    benchmark_series
], axis=1)

all_series=all_series.loc[:, ~all_series.columns.duplicated()]


In [ ]:
#Calculations
mode='standard'

benchmark_minus_indices_week          = create_spreads(indices_series, benchmark_series, time_frame=time_frame_week,mode=mode)
benchmark_minus_sectors_week          = create_spreads(sectors_series, benchmark_series, time_frame=time_frame_week,mode=mode)
benchmark_minus_industries_week       = create_spreads(industries_series, benchmark_series, time_frame=time_frame_week,mode=mode)
benchmark_minus_top_spy_holdings_week = create_spreads(top_spy_holdings_series, benchmark_series, time_frame=time_frame_week,mode=mode)
benchmark_minus_top_qqq_holdings_week = create_spreads(top_qqq_holdings_series, benchmark_series, time_frame=time_frame_week,mode=mode)
benchmark_minus_top_dia_holdings_week = create_spreads(top_dia_holdings_series, benchmark_series, time_frame=time_frame_week,mode=mode)
benchmark_minus_top_xlk_holdings_week = create_spreads(top_xlk_holdings_series, benchmark_series, time_frame=time_frame_week,mode=mode)
benchmark_minus_top_xlf_holdings_week = create_spreads(top_xlf_holdings_series, benchmark_series, time_frame=time_frame_week,mode=mode)

benchmark_minus_all_series_week= pd.concat([
    benchmark_minus_indices_week,
    benchmark_minus_sectors_week,
    benchmark_minus_industries_week,
    benchmark_minus_top_spy_holdings_week,
    benchmark_minus_top_qqq_holdings_week,
    benchmark_minus_top_dia_holdings_week,
    benchmark_minus_top_xlk_holdings_week,
    benchmark_minus_top_xlf_holdings_week
], axis=1)

benchmark_minus_all_series_week=benchmark_minus_all_series_week.loc[:, ~benchmark_minus_all_series_week.columns.duplicated()]

benchmark_minus_indices_short          = create_spreads(indices_series, benchmark_series, time_frame=time_frame_short,mode=mode)
benchmark_minus_sectors_short          = create_spreads(sectors_series, benchmark_series, time_frame=time_frame_short,mode=mode)
benchmark_minus_industries_short       = create_spreads(industries_series, benchmark_series, time_frame=time_frame_short,mode=mode)
benchmark_minus_top_spy_holdings_short = create_spreads(top_spy_holdings_series, benchmark_series, time_frame=time_frame_short,mode=mode)
benchmark_minus_top_qqq_holdings_short = create_spreads(top_qqq_holdings_series, benchmark_series, time_frame=time_frame_short,mode=mode)
benchmark_minus_top_dia_holdings_short = create_spreads(top_dia_holdings_series, benchmark_series, time_frame=time_frame_short,mode=mode)
benchmark_minus_top_xlk_holdings_short = create_spreads(top_xlk_holdings_series, benchmark_series, time_frame=time_frame_short,mode=mode)
benchmark_minus_top_xlf_holdings_short = create_spreads(top_xlf_holdings_series, benchmark_series, time_frame=time_frame_short,mode=mode)

benchmark_minus_all_series_short= pd.concat([
    benchmark_minus_indices_short,
    benchmark_minus_sectors_short,
    benchmark_minus_industries_short,
    benchmark_minus_top_spy_holdings_short,
    benchmark_minus_top_qqq_holdings_short,
    benchmark_minus_top_dia_holdings_short,
    benchmark_minus_top_xlk_holdings_short,
    benchmark_minus_top_xlf_holdings_short
], axis=1)

benchmark_minus_all_series_short=benchmark_minus_all_series_short.loc[:, ~benchmark_minus_all_series_short.columns.duplicated()]

benchmark_minus_indices_mid          = create_spreads(indices_series, benchmark_series, time_frame=time_frame_mid,mode=mode)
benchmark_minus_sectors_mid          = create_spreads(sectors_series, benchmark_series, time_frame=time_frame_mid,mode=mode)
benchmark_minus_industries_mid        = create_spreads(industries_series, benchmark_series, time_frame=time_frame_mid,mode=mode)
benchmark_minus_top_spy_holdings_mid    = create_spreads(top_spy_holdings_series, benchmark_series, time_frame=time_frame_mid,mode=mode)
benchmark_minus_top_qqq_holdings_mid    = create_spreads(top_qqq_holdings_series, benchmark_series, time_frame=time_frame_mid,mode=mode)
benchmark_minus_top_dia_holdings_mid    = create_spreads(top_dia_holdings_series, benchmark_series, time_frame=time_frame_mid,mode=mode)
benchmark_minus_top_xlk_holdings_mid    = create_spreads(top_xlk_holdings_series, benchmark_series, time_frame=time_frame_mid,mode=mode)
benchmark_minus_top_xlf_holdings_mid    = create_spreads(top_xlf_holdings_series, benchmark_series, time_frame=time_frame_mid,mode=mode)

benchmark_minus_all_series_mid= pd.concat([
    benchmark_minus_indices_mid,
    benchmark_minus_sectors_mid,
    benchmark_minus_industries_mid,
    benchmark_minus_top_spy_holdings_mid,
    benchmark_minus_top_qqq_holdings_short,
    benchmark_minus_top_dia_holdings_mid,
    benchmark_minus_top_xlk_holdings_mid,
    benchmark_minus_top_xlf_holdings_mid
], axis=1)

benchmark_minus_all_series_mid=benchmark_minus_all_series_mid.loc[:, ~benchmark_minus_all_series_mid.columns.duplicated()]


In [ ]:
#Graph Market Caps
def plot_market_caps(info):
    market_caps = pd.DataFrame(info[['Symbol','Market Cap']])
    #market_caps = market_caps.sort_values(by='Market Cap',ascending=False)
    market_caps = market_caps.sort_values(by='Market Cap')
    market_caps['Log Market Cap'] = np.log(market_caps['Market Cap'])
    percentiles = np.percentile(market_caps['Log Market Cap'], [60, 90])

    def categorize(market_cap):
        if market_cap <= percentiles[0]:
            return 'Small-Cap'
        elif market_cap <= percentiles[1]:
            return 'Mid-Cap'
        else:
            return 'Large-Cap'

    market_caps['category'] = market_caps['Log Market Cap'].apply(categorize)
    value_counts = market_caps['category'].value_counts()
    fig = px.bar(market_caps, x='Symbol', y='Market Cap', 
                labels={'Ticker': 'Symbol', 'Market Cap': 'Market Cap (Billions USD)'},
                title='Market Capitalizations Companies')

    # Customize layout if needed
    fig.update_layout(
        xaxis_title="Ticker",
        yaxis_title="Market Cap (Billions USD)"
    )
    mid_to_large= len(market_caps) - market_caps.tail(value_counts['Large-Cap']).count()['Symbol']+ 2.5
    small_to_mid= market_caps.tail(value_counts['Small-Cap']).count()['Symbol'] - .5
    market_caps[market_caps['category'] == 'Mid-Cap']

    # Add vertical lines to separate the caps
    fig.add_vline(x=small_to_mid, line=dict(color="Red", width=2, dash="dashdot"), annotation_text="Small to Mid", annotation_position="top left")
    fig.add_vline(x=mid_to_large, line=dict(color="Blue", width=2, dash="dashdot"), annotation_text="Mid to Large", annotation_position="top left")

    fig.show()

plot_market_caps(sp500_info)
plot_market_caps(nasdaq_info)
plot_market_caps(dow_info)
plot_market_caps(xlk_info)
plot_market_caps(xlf_info)
plot_market_caps(xli_info)
plot_market_caps(xlv_info)
plot_market_caps(xlu_info)
plot_market_caps(xlp_info)
plot_market_caps(xlc_info)
plot_market_caps(xlb_info)
plot_market_caps(xlre_info)
plot_market_caps(xle_info)



In [ ]:
#Find Oversold Investments relative to S&P 500
filtered_assets = filter_assets_by_positive_spread_std(benchmark_minus_all_series_mid)
filtered_assets = filtered_assets[filtered_assets]
#Create a Plotly Table trace
fig = go.Figure(data=[go.Table(
    header=dict(values=['Date', 'Boolean Value']),
    cells=dict(values=[filtered_assets.index, filtered_assets], align='left'))
])

fig.show()


In [ ]:
#Analyze the potential Profitability of an asset
TICKER = 'XLE'
SPREAD = 'Benchmark_minus_' + TICKER
create_spread_plot(benchmark_minus_all_series_week[SPREAD]).show()
create_spread_plot(benchmark_minus_all_series_short[SPREAD]).show()
create_spread_plot(benchmark_minus_all_series_mid[SPREAD]).show()
plot_risk_adjusted_returns(all_series[TICKER],time_frame_short).show()
plot_risk_adjusted_returns(all_series[TICKER],time_frame_mid).show()


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# Assuming you have xlk_holdings_series and xlf_holdings_series as DataFrames

# Define the risk-free rate (assuming it's constant, e.g., 0.01 for 1%)
risk_free_rate = 0.01 / 252  # Daily risk-free rate

# Define the function to calculate the Sortino ratio
def calculate_sortino_ratio(returns):
    excess_returns = returns - risk_free_rate
    downside_deviation = excess_returns[excess_returns < 0].std()
    sortino_ratio = excess_returns.mean() / downside_deviation if downside_deviation != 0 else np.nan
    return sortino_ratio

# Define the function to calculate rolling Sortino ratios
def calculate_risk_adjusted_returns(df, time_frame):
    daily_returns = df.pct_change()
    rolling_sortino_ratio = daily_returns.rolling(window=time_frame).apply(calculate_sortino_ratio)
    return rolling_sortino_ratio

# Define the rolling window size
time_frame = 50

# Calculate rolling Sortino ratios for xlk_holdings_series and xlf_holdings_series
xlk_sortino_ratios = calculate_risk_adjusted_returns(xlk_holdings_series, time_frame)
xlf_sortino_ratios = calculate_risk_adjusted_returns(xlf_holdings_series, time_frame)

# Calculate the mean Sortino ratio for each stock
mean_sortino_xlk = xlk_sortino_ratios.mean()
mean_sortino_xlf = xlf_sortino_ratios.mean()

# Filter out periods where the Sortino ratio is below the mean for each stock
filtered_xlk = xlk_sortino_ratios.apply(lambda x: x[x >= mean_sortino_xlk[x.name]])
filtered_xlf = xlf_sortino_ratios.apply(lambda x: x[x >= mean_sortino_xlf[x.name]])

# Plot the results using Plotly

# Plot for filtered XLK holdings
fig_xlk = go.Figure()

for column in filtered_xlk.columns:
    fig_xlk.add_trace(go.Scatter(x=filtered_xlk.index, y=filtered_xlk[column], mode='lines', name=column))

fig_xlk.update_layout(
    title='50-Day Rolling Sortino Ratios for Filtered XLK Holdings',
    xaxis_title='Date',
    yaxis_title='50-Day Rolling Sortino Ratio',
    legend_title='Ticker',
    template='plotly_dark'
)

# Plot for filtered XLF holdings
fig_xlf = go.Figure()

for column in filtered_xlf.columns:
    fig_xlf.add_trace(go.Scatter(x=filtered_xlf.index, y=filtered_xlf[column], mode='lines', name=column))

fig_xlf.update_layout(
    title='50-Day Rolling Sortino Ratios for Filtered XLF Holdings',
    xaxis_title='Date',
    yaxis_title='50-Day Rolling Sortino Ratio',
    legend_title='Ticker',
    template='plotly_dark'
)

fig_xlk.show()
fig_xlf.show()


